
```

```



#**B.Manipulasi Data Dengan Apache Spark SQL**
#**[SELECT & FUNCTION Statment ]**

##**1. Hadoop & PySpark initialization di Google Colab**

*PySpark* adalah *interface Python* untuk *Apache Spark*. Penggunaan utama *PySpark* adalah untuk bekerja dengan data dalam Bigdata dan untuk membuat pipeline data.

Walaupun *Apache Spark* mendudukung Big Data, sebagai awal pembelajaran tidak perlu menggunakan data yang besar untuk mendapatkan manfaat dari PySpark. Kita bisa temukan bahwa SparkSQL adalah tools yang bagus untuk melakukan analisis data. Penggunaan library Panda menjadi lambat dengan data yang besar Sumber tentang Apache Spark http://spark.apache.org/docs/latest/api/python/



In [ ]:
#Init Hadoop & Spark
#=====Begin
!pip install pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PySparkSQL").getOrCreate()
print("Spark Session initialized.")
#=====End

In [ ]:
# Download datasets
!wget -O ms_produk.csv https://github.com/rahmadsa/dataset/blob/main/spark/ms_produk.csv?raw=true
!wget -O ms_pelanggan.csv https://github.com/rahmadsa/dataset/blob/main/spark/ms_pelanggan.csv?raw=true
!wget -O tr_penjualan.csv https://github.com/rahmadsa/dataset/blob/main/spark/tr_penjualan.csv?raw=true
!wget -O students.csv https://github.com/rahmadsa/dataset/blob/main/spark/students.csv?raw=true

# Load datasets into Spark DataFrames
ms_produk_df = spark.read.csv('ms_produk.csv', header=True, inferSchema=True)
ms_pelanggan_df = spark.read.csv('ms_pelanggan.csv', header=True, inferSchema=True)
tr_penjualan_df = spark.read.csv('tr_penjualan.csv', header=True, inferSchema=True)
students_df = spark.read.csv('students.csv', header=True, inferSchema=True)

# Function to clean and rename columns for a DataFrame using toDF
def clean_df_columns(df):
    old_cols = df.columns
    new_cols = [col.strip().replace(' ', '_') for col in old_cols]
    return df.toDF(*new_cols)

# Clean column names for all DataFrames
ms_produk_df = clean_df_columns(ms_produk_df)
ms_pelanggan_df = clean_df_columns(ms_pelanggan_df)
tr_penjualan_df = clean_df_columns(tr_penjualan_df)
students_df = clean_df_columns(students_df)

# Register DataFrames as temporary views
# Drop views first for robustness against caching issues
try:
    spark.catalog.dropTempView('ms_produk')
    spark.catalog.dropTempView('ms_pelanggan')
    spark.catalog.dropTempView('tr_penjualan')
    spark.catalog.dropTempView('students')
except Exception as e:
    print(f"Could not drop temp views (might not exist yet): {e}")

ms_produk_df.createOrReplaceTempView('ms_produk')
ms_pelanggan_df.createOrReplaceTempView('ms_pelanggan')
tr_penjualan_df.createOrReplaceTempView('tr_penjualan')
students_df.createOrReplaceTempView('students')

print("Datasets loaded and temporary views created.")

# Verify the schema of the students temporary view
print("Schema of 'students' temporary view (via spark.table):")
spark.table('students').printSchema()

In [ ]:
# Display schema and first few rows to verify
print("ms_produk schema:")
ms_produk_df.printSchema()
ms_produk_df.show(5)

print("ms_pelanggan schema:")
ms_pelanggan_df.printSchema()
ms_pelanggan_df.show(5)

print("tr_penjualan schema:")
tr_penjualan_df.printSchema()
tr_penjualan_df.show(5)

print("students schema:")
students_df.printSchema()
students_df.show(5)

Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/ms_produk.csv?raw=true"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

#**2. Manipulasi Data Dengan SELECT Statment**
##**2.1. Mengambil Seluruh Kolom dalam suatu Tabel**


```
   select * from ms_produk;
```

In [ ]:
spark.sql("select * from ms_produk;").show()

##**2.2. Mengambil Satu Kolom dari Tabel**

In [ ]:
spark.sql("select nama_produk from ms_produk;").show()

## **2.3. Mengambil Lebih dari Satu Kolom dari Tabel**
```
   SELECT nama_produk, harga FROM ms_produk;
```

In [ ]:
spark.sql("SELECT nama_produk, harga FROM ms_produk;").show()

##**2.4. Membatasi Pengambilan Jumlah Row Data**

```
  SELECT nama_produk, harga FROM ms_produk LIMIT 5;
```

In [ ]:
spark.sql("SELECT nama_produk, harga FROM ms_produk LIMIT 5;").show()

##**2.5 Penggunaan SELECT DISTINCT statement**

Lakukan manipulasi data dari dataset pada tautan berikut:
"https://github.com/rahmadsa/dataset/blob/main/spark/ms_penggan.csv"
dengan query sebagai berikut.

```
SELECT DISTINCT nama_customer, alamat FROM ms_pelanggan;
```

In [ ]:
spark.sql("SELECT DISTINCT nama_customer, alamat FROM ms_pelanggan;").show()

##**2.6. Menggunakan Prefix pada Nama Kolom**

```
SELECT ms_produk.kode_produk FROM ms_Produk;
```
Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/ms_produk.csv"
dengan query sebagai berikut.

In [ ]:
spark.sql("SELECT ms_produk.kode_produk FROM ms_Produk;").show()

##**2.7. Menggunakan Alias pada Kolom**

Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/ms_produk.csv"
dengan query sebagai berikut.

```
SELECT no_urut AS nomor, nama_produk AS nama FROM ms_produk;
```


In [ ]:
spark.sql("SELECT no_urut AS nomor, nama_produk AS nama FROM ms_produk;").show()

##**2.8.Menggabungkan Prefix dan Alias**
```
SELECT ms_produk.harga AS harga_jual FROM ms_produk;
```

In [ ]:
spark.sql("SELECT ms_produk.harga AS harga_jual FROM ms_produk;").show()

##**2.9. Menggunakan Alias pada Tabel**

```
SELECT * FROM ms_produk t2;
```

In [ ]:
spark.sql("SELECT * FROM ms_produk t2;").show()

##**2.10. Prefix dengan Alias Tabel**

```
SELECT t2.nama_produk, t2.harga FROM ms_produk t2
```

In [ ]:
spark.sql("SELECT t2.nama_produk, t2.harga FROM ms_produk t2;").show()

##**2.11. Menggunakan WHERE**

```
SELECT * FROM ms_produk WHERE nama_produk='Tas Travel Organizer';
```

In [ ]:
spark.sql("SELECT * FROM ms_produk WHERE nama_produk='Tas Travel Organizer';").show()

#**2.12. Menggunakan Operand OR**

```
SELECT * FROM ms_produk WHERE nama_produk='Gantungan Kunci' OR nama_produk='Tas Travel Organizer'OR nama_produk='Flashdisk 64 GB' ;
```


In [ ]:
spark.sql("SELECT * FROM ms_produk WHERE nama_produk='Gantungan Kunci' OR nama_produk='Tas Travel Organizer' OR nama_produk='Flashdisk 64 GB';").show()

##**2.13. Filter untuk Angka**
```
SELECT * FROM ms_produk WHERE harga>50000;
```

In [ ]:
spark.sql("SELECT * FROM ms_produk WHERE harga>50000;").show()

##**2.14. Menggunakan Operand AND**
```
SELECT * FROM ms_produk WHERE nama_produk='Gantungan Kunci' AND harga<50000;
```

In [ ]:
spark.sql("SELECT * FROM ms_produk WHERE nama_produk='Gantungan Kunci' AND harga<50000;").show()

###**2.15. Proyek dari Cabang A**
Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/tr_penjualan.csv"
dengan query - query sebagai berikut.

```
SELECT kode_pelanggan, nama_produk, qty, harga, qty*harga AS total FROM tr_penjualan WHERE qty*harga>=100000 ORDER BY total DESC;```
```

In [ ]:
spark.sql("SELECT kode_pelanggan, nama_produk, qty, harga, qty*harga AS total FROM tr_penjualan WHERE qty*harga>=100000 ORDER BY total DESC;").show()

#**3. Manipulasi Data Dengan Fungsi**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali. Fungsi ini membantu menyederhanakan dan mengoptimalkan kueri SQL, serta memungkinkan penggunaan kembali kode

## **3.1. Fungsi Skalar Matematika - ABS()**

Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/students.csv"
dengan query sebagai berikut.

```
SELECT StudentID, FirstName, LastName, Semester1, Semester2, ABS(MarkGrowth) AS MarkGrowth
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LastName, ABS(MarkGrowth) AS MarkGrowth FROM students;").show()

## **3.2. Fungsi Skalar Matematika - CEILING()**
Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/students.csv"
dengan query - query sebagai berikut.

```
SELECT StudentID, FirstName, LastName, CEILING(Semester1) AS Semester1, CEILING(Semester2) AS Semester2,MarkGrowth
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LastName, CEILING(Semester1) AS Semester1, CEILING(Semester2) AS Semester2, MarkGrowth FROM students;").show()

##**3.3. Fungsi Skalar Matematika - FLOOR()**

```
SELECT StudentID, FirstName, LastName, FLOOR(Semester1) AS Semester1, FLOOR(Semester2) AS Semester2, MarkGrowth
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LastName, FLOOR(Semester1) AS Semester1, FLOOR(Semester2) AS Semester2, MarkGrowth FROM students;").show()

##**3.4. Fungsi Skalar Matematika - ROUND()**

```
SELECT StudentID, FirstName, LastName, ROUND(Semester1,1) AS Semester1, ROUND(Semester2,0) AS Semester2, MarkGrowth
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LastName, ROUND(Semester1,1) AS Semester1, ROUND(Semester2,0) AS Semester2, MarkGrowth FROM students;").show()

##**3.5. Fungsi Skalar Matematika - SQRT( )**
```
SELECT StudentID, FirstName, LastName, SQRT(Semester1) AS Semester1, Semester2, MarkGrowth
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LastName, SQRT(Semester1) AS Semester1, Semester2, MarkGrowth FROM students;").show()

##**3.6 Fungsi Text - CONCAT( )**
```
SELECT StudentID, CONCAT(FirstName, LastName) AS Name, Semester1, Semester2, MarkGrowth
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, CONCAT(FirstName, LastName) AS Name, Semester1, Semester2, MarkGrowth FROM students;").show()

##**3.7. Fungsi Text - SUBSTRING_INDEX( )**
```
SELECT StudentID, SUBSTRING_INDEX(Email, '@', 1) AS Name
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, SUBSTRING_INDEX(Email, '@', 1) AS Name FROM students;").show()

##**3.8. Fungsi Text - SUBSTR( )**
```
SELECT StudentID, SUBSTR(FirstName, 2, 3) AS Initial
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, SUBSTR(FirstName, 2, 3) AS Initial FROM students;").show()

## **3.9. Fungsi Text - LENGTH( )**
```
SELECT StudentID, FirstName, LENGTH(FirstName) AS Total_Char
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LENGTH(FirstName) AS Total_Char FROM students;").show()

##**3.10. Fungsi Text - REPLACE( )**
```
SELECT StudentID, Email, REPLACE(Email,'yahoo', 'gmail') AS New_Email
FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, Email, REPLACE(Email,'yahoo', 'gmail') AS New_Email FROM students;").show()

## **3.11. Fungsi Aggregate - SUM()**
```
SELECT SUM(Semester1) AS Total_1, SUM(Semester2) AS Total_2
FROM students;
```

In [ ]:
spark.sql("SELECT SUM(Semester1) AS Total_1, SUM(Semester2) AS Total_2 FROM students;").show()

## **3.12. Fungsi Aggregate - COUNT()**
```
SELECT COUNT(FirstName) AS Total_Student
FROM students;
```

In [ ]:
spark.sql("SELECT COUNT(FirstName) AS Total_Student FROM students;").show()

##**3.13. Fungsi Aggregate - AVG( )**
```
SELECT AVG(Semester1) AS AVG_1, AVG(Semester2) AS AVG_2
FROM students;
```

In [ ]:
spark.sql("SELECT AVG(Semester1) AS AVG_1, AVG(Semester2) AS AVG_2 FROM students;").show()

##**3.14. Tugas Praktek-2.1**
```
SELECT StudentID, FirstName, LastName, MOD(Semester1,2) AS Semester1 , Semester2, EXP(MarkGrowth) FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, FirstName, LastName, MOD(Semester1,2) AS Semester1 , Semester2, EXP(MarkGrowth) FROM students;").show()

## **3.15. Tugas Praktek-2.2**
```
SELECT StudentID, UPPER(FirstName) AS FirstName, LOWER(LastName) AS LastName FROM students;
```

In [ ]:
spark.sql("SELECT StudentID, UPPER(FirstName) AS FirstName, LOWER(LastName) AS LastName FROM students;").show()